# 05 — Cell type annotation (learning + validation)

**Input**: `04_integrated_clustered.h5ad`  
**Output**: `05_annotated.h5ad`

Two-track approach:
1. **Learning track**: score each cell against canonical markers, assign per-cluster identities
2. **Validation track**: compare against Wu et al.'s published `celltype_major` annotations

If both agree, we have confident cell type labels. Where they disagree, inspect the specific markers.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src import annotation as anno

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 5))

adata = sc.read_h5ad(cfg.H5AD_INTEGRATED)
cluster_key = f'leiden_r{cfg.LEIDEN_DEFAULT}'
print(f'Using cluster resolution: {cluster_key} ({adata.obs[cluster_key].nunique()} clusters)')

## Learning track: marker-based scoring

In [ ]:
# Score each cell against canonical marker gene sets (see config.py)
adata = anno.score_cell_type_markers(adata, cfg.CELL_TYPE_MARKERS)
score_cols = [f'{ct}_score' for ct in cfg.CELL_TYPE_MARKERS]

In [ ]:
# Visualize scores on the UMAP. One panel per cell type.
sc.pl.umap(adata, color=score_cols, ncols=3, wspace=0.3, show=False)
plt.savefig(cfg.FIGURES_DIR / 'umap_marker_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dotplot of marker genes across clusters: gold standard for checking identities
all_markers = sum(cfg.CELL_TYPE_MARKERS.values(), [])
all_markers = [g for g in all_markers if g in adata.var_names]
sc.pl.dotplot(
    adata,
    var_names=all_markers,
    groupby=cluster_key,
    standard_scale='var',
    show=False,
)
plt.savefig(cfg.FIGURES_DIR / 'dotplot_markers_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-cluster top cell type assignments
cluster_scores = anno.assign_cluster_identities(
    adata,
    cfg.CELL_TYPE_MARKERS,
    cluster_key=cluster_key,
)
cluster_scores

In [ ]:
# Build cluster_to_celltype mapping. Start with top_cell_type column, then manually override edge cases after inspecting the dotplot above.
cluster_to_celltype = {
    '0':  'T_cells',
    '1':  'T_cells',
    '2':  'Normal_Epithelial',        # ← manual override: basal score high but DCN+ in dotplot
    '3':  'T_cells',
    '4':  'Cancer_Epithelial',        # luminal-dominant
    '5':  'Cancer_Epithelial',        # luminal, mixed with normal
    '6':  'T_cells',
    '7':  'Cancer_Epithelial',        # luminal+basal co-expression
    '8':  'Myeloid',                  # ← manual override: dotplot shows CD68/LYZ/CSF1R
    '9':  'Myeloid',                  # ← manual override: same as 8
    '10': 'B_cells',
    '11': 'Plasma_cells',
    '12': 'Myeloid',
    '13': 'Cancer_Epithelial',        # luminal score moderate, basal component
    '14': 'Cancer_Epithelial',        # luminal+basal mixed
    '15': 'Fibroblasts_CAFs',
    '16': 'Perivascular',
    '17': 'Cancer_Epithelial',        # luminal
    '18': 'Plasma_cells',
    '19': 'Cancer_Epithelial',        # luminal
    '20': 'Cancer_Epithelial',        # luminal
    '21': 'Cancer_Epithelial',        # luminal
}

In [ ]:
# Run the annotation cell with the corrected dictionary

adata = anno.apply_cluster_annotation(
    adata,
    cluster_to_celltype,
    cluster_key=cluster_key,
    annotation_key='celltype_predicted',
)

print('Predicted cell type counts:')
print(adata.obs['celltype_predicted'].value_counts())

In [ ]:
# UMAP with our predicted annotations
sc.pl.umap(
    adata,
    color='celltype_predicted',
    legend_loc='on data',
    legend_fontsize=7,
    show=False,
)
plt.savefig(cfg.FIGURES_DIR / 'umap_celltype_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Validation track: compare against Wu et al. labels

In [ ]:
if cfg.CELLTYPE_COLUMN_AUTHORS in adata.obs.columns:
    ct = anno.compare_annotations(
        adata,
        col_authors=cfg.CELLTYPE_COLUMN_AUTHORS,
        col_predicted='celltype_predicted',
    )
    print('Crosstab (rows=authors, cols=predicted):')
    display(ct)
else:
    print(f'No column "{cfg.CELLTYPE_COLUMN_AUTHORS}" in metadata.')

In [ ]:
# Side-by-side UMAP comparison
import matplotlib.pyplot as plt
import scanpy as sc

# Define shared color palette — same color for same biological population
# Wu's labels on left, your labels on right
shared_colors = {
    # Wu labels (celltype_major)
    'B-cells':            '#1f77b4',   # blue
    'CAFs':               '#ff7f0e',   # orange  
    'Cancer Epithelial':  '#2ca02c',   # green
    'Endothelial':        '#d62728',   # red
    'Myeloid':            '#9467bd',   # purple
    'Normal Epithelial':  '#8c564b',   # brown
    'PVL':                '#e377c2',   # pink
    'Plasmablasts':       '#bcbd22',   # yellow-green
    'T-cells':            '#17becf',   # cyan
    # Your labels (celltype_predicted) — same colors where equivalent
    'B_cells':            '#1f77b4',   # blue = B-cells
    'Cancer_Epithelial':  '#2ca02c',   # green = Cancer Epithelial
    'Fibroblasts_CAFs':   '#ff7f0e',   # orange = CAFs
    'Myeloid':            '#9467bd',   # purple = Myeloid (yours catches endothelial too)
    'Normal_Epithelial':  '#8c564b',   # brown = Normal Epithelial
    'Perivascular':       '#e377c2',   # pink = PVL
    'Plasma_cells':       '#bcbd22',   # yellow-green = Plasmablasts
    'T_cells':            '#17becf',   # cyan = T-cells
}

fig, axs = plt.subplots(1, 2, figsize=(16, 7))

# Left: Wu's labels
wu_categories = adata.obs['celltype_major'].cat.categories.tolist()
wu_palette = [shared_colors.get(c, '#aaaaaa') for c in wu_categories]

sc.pl.umap(
    adata,
    color='celltype_major',
    palette=wu_palette,
    legend_loc='right margin',
    legend_fontsize=9,
    title="Wu et al. labels (celltype_major)",
    show=False,
    ax=axs[0],
)

# Right: your predicted labels
pred_categories = adata.obs['celltype_predicted'].cat.categories.tolist()
pred_palette = [shared_colors.get(c, '#aaaaaa') for c in pred_categories]

sc.pl.umap(
    adata,
    color='celltype_predicted',
    palette=pred_palette,
    legend_loc='right margin',
    legend_fontsize=9,
    title="Independent annotation (celltype_predicted)",
    show=False,
    ax=axs[1],
)

plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'umap_authors_vs_predicted_matched_colors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: umap_authors_vs_predicted_matched_colors.png')

In [ ]:
adata.write_h5ad(cfg.H5AD_ANNOTATED, compression='gzip')
print(f'Saved: {cfg.H5AD_ANNOTATED} ({cfg.H5AD_ANNOTATED.stat().st_size / 1e6:.1f} MB)')

---

**Next**: `06_arc_expression.ipynb` — the core biological analysis.